# Data loading

Link to dataset at: <https://doi.org/10.7910/DVN/PR4L8P>

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, f_oneway
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from sklearn.model_selection import GroupKFold
import statsmodels.formula.api as smf

df = pd.read_csv("CCES22_Common_OUTPUT_vv_topost.csv", low_memory=False)

# Exploratory data analysis

## Basic dataset information

In [2]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 707 entries, Unnamed: 0 to sourcevar
dtypes: float64(449), int64(89), str(169)
memory usage: 323.6 MB
None


## Correlation between input and target variables.

Higher value of Cramer's V represents stronger association between the input and target variables. Lower p-value means the result is statistically significant based on the input variables.

In [3]:
cat = ["gender4", "CC22_320d", "CC22_320g", "CC22_363", "presvote20post", "newsint"]
num = ["birthyr"]

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, p, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt((chi2 / n) / (min(ct.shape) - 1))
    return v, p

print("\nCorrelation between input and target features:")
for target in ["CC22_365", "CC22_366"]:
    for c in cat:
        v, p = cramers_v(df[c], df[target])
        print(target, c, f"V={v:.2f}, p={p:.3g}")

    groups = [g["birthyr"].values for _, g in df.groupby(target)]
    print(target, "birthyr", f_oneway(*groups))


Correlation between input and target features:
CC22_365 gender4 V=0.08, p=1.41e-150
CC22_365 CC22_320d V=0.20, p=0
CC22_365 CC22_320g V=0.23, p=0
CC22_365 CC22_363 V=0.27, p=0
CC22_365 presvote20post V=0.42, p=0
CC22_365 newsint V=0.23, p=0
CC22_365 birthyr F_onewayResult(statistic=np.float64(563.6584556331454), pvalue=np.float64(0.0))
CC22_366 gender4 V=0.08, p=1.65e-145
CC22_366 CC22_320d V=0.22, p=0
CC22_366 CC22_320g V=0.19, p=0
CC22_366 CC22_363 V=0.27, p=0
CC22_366 presvote20post V=0.40, p=0
CC22_366 newsint V=0.21, p=0
CC22_366 birthyr F_onewayResult(statistic=np.float64(578.0056621441535), pvalue=np.float64(0.0))


## Distribution of responses (dataset rows) per state

In [4]:
print(df["inputstate"].value_counts(dropna=False))

inputstate
6     4933
48    4750
12    4426
36    3718
42    3106
39    2515
17    2231
37    1993
13    1988
26    1953
51    1646
4     1625
34    1573
47    1312
29    1297
18    1282
53    1275
55    1253
25    1096
8     1042
24    1018
27     993
45     978
41     970
21     946
1      941
32     758
22     717
40     686
9      597
19     595
5      546
49     503
28     482
20     463
54     458
35     415
31     347
23     323
16     304
33     286
30     224
10     223
15     208
44     197
46     171
11     165
2      135
38     129
50     112
56      96
Name: count, dtype: int64


## Not all states had races for positions

In [5]:
sen_races = df[df["SenCand1Name"].notna()].reset_index(drop=True)
gov_races = df[df["GovCand1Name"].notna()].reset_index(drop=True)
print(f"States with Sen Races: {sen_races['inputstate'].nunique()}")
print(f"States with Gov Races: {gov_races['inputstate'].nunique()}")

States with Sen Races: 34
States with Gov Races: 36


# Data cleaning

Drop rows missing the input features:

In [6]:
features = [
    "gender4",
    "CC22_320d",
    "CC22_320g",
    "CC22_363",
    "presvote20post",
    "newsint",
    "birthyr",
]

race_check_cols = ["SenCand1Name", "GovCand1Name"]
clean_df = df.dropna(subset=["caseid", "inputstate", "commonweight"] + features).copy()
print("Rows after dropping missing features:", len(clean_df))

Rows after dropping missing features: 58126


Remove rows where there was no race in the seat:

In [7]:
long_df = clean_df.melt(
    id_vars=["caseid", "inputstate", "commonweight"] + features + race_check_cols,
    value_vars=["CC22_365", "CC22_366"],
    var_name="seat",
    value_name="candidate_slot",
)

has_senate_race = (long_df["seat"] == "CC22_365") & long_df["SenCand1Name"].notna() & (long_df["SenCand1Name"].astype(str).str.strip() != "")
has_gov_race = (long_df["seat"] == "CC22_366") & long_df["GovCand1Name"].notna() & (long_df["GovCand1Name"].astype(str).str.strip() != "")

long_df = long_df[has_senate_race | has_gov_race].copy()

long_df = long_df.drop(columns=race_check_cols)

Remove rows with invalid candidate choices e.g. "I'm not sure", "Other":

In [8]:
valid_slots = [1, 2, 3]
model_df = long_df[long_df["candidate_slot"].isin(valid_slots)].copy()
model_df["y"] = (model_df["candidate_slot"] - 1).astype(int)
print("Rows in stacked model_df before level pruning:", len(model_df))

cat_cols = ["gender4", "CC22_320d", "CC22_320g", "CC22_363", "presvote20post", "newsint"]

for col in cat_cols:
    ct = pd.crosstab(model_df[col], model_df["y"])
    valid_levels = ct[(ct >= 5).all(axis=1)].index
    model_df = model_df[model_df[col].isin(valid_levels)].copy()

assert len(model_df) > 0, "model_df is empty! Check valid_slots and crosstabs."

active_cats = [col for col in cat_cols if model_df[col].nunique() > 1]
model_df["birthyr_std"] = (model_df["birthyr"] - model_df["birthyr"].mean()) / model_df["birthyr"].std()

rhs = " + ".join([f"C({col})" for col in active_cats] + ["birthyr_std"])
formula = f"y ~ {rhs}"
print(f"Fitting on {len(model_df)} rows with formula: {formula}")

Rows in stacked model_df before level pruning: 63255
Fitting on 61892 rows with formula: y ~ C(gender4) + C(CC22_320d) + C(CC22_320g) + C(CC22_363) + C(presvote20post) + C(newsint) + birthyr_std


# Data modelling

Apply multinomial regression on the transformed dataset to model the share distribution and predict vote shares for the seats.

In [9]:
result = smf.mnlogit(formula, data=model_df).fit(
    method="bfgs",
    maxiter=500,
    cov_type="cluster",
    cov_kwds={"groups": model_df["caseid"]},
)

probs = result.predict(model_df).reset_index(drop=True)
probs.columns = ["cand_1_prob", "cand_2_prob", "cand_3_prob"]
pred_df = pd.concat([model_df[["inputstate", "seat", "commonweight"]].reset_index(drop=True), probs], axis=1)

state_seat_shares = (
    pred_df.groupby(["inputstate", "seat"], as_index=False)
    .apply(
        lambda g: pd.Series({
            "Cand1_Share": np.average(g["cand_1_prob"], weights=g["commonweight"]),
            "Cand2_Share": np.average(g["cand_2_prob"], weights=g["commonweight"]),
            "Cand3_Share": np.average(g["cand_3_prob"], weights=g["commonweight"]),
        }),
        include_groups=False,
    )
)

print(state_seat_shares.head(20))

Optimization terminated successfully.
         Current function value: 0.288005
         Iterations: 252
         Function evaluations: 253
         Gradient evaluations: 253
    inputstate      seat  Cand1_Share  Cand2_Share  Cand3_Share
0            1  CC22_365     0.413503     0.580962     0.005536
1            1  CC22_366     0.428367     0.566285     0.005348
2            2  CC22_365     0.776101     0.218242     0.005658
3            2  CC22_366     0.492926     0.500606     0.006467
4            4  CC22_365     0.477534     0.516506     0.005960
5            4  CC22_366     0.470772     0.523289     0.005939
6            5  CC22_365     0.480929     0.512689     0.006382
7            5  CC22_366     0.458174     0.535535     0.006292
8            6  CC22_365     0.624925     0.369390     0.005685
9            6  CC22_366     0.617379     0.377058     0.005563
10           8  CC22_365     0.560322     0.435166     0.004511
11           8  CC22_366     0.562544     0.432932     0.

# Model evaluation

Perform k-fold cross validation along an 80-20 split:

In [10]:
gkf = GroupKFold(n_splits=5)
oof_probs = np.zeros((len(model_df), len(valid_slots)))

gkf_state = GroupKFold(n_splits=5)
state_oof_probs = np.zeros((len(model_df), len(valid_slots)))

for train_idx, test_idx in gkf.split(model_df, model_df["y"], groups=model_df["caseid"]):
    train_df = model_df.iloc[train_idx]
    test_df = model_df.iloc[test_idx]

    fold_res = smf.mnlogit(formula, data=train_df).fit(method="bfgs", maxiter=500, disp=False)
    oof_probs[test_idx] = fold_res.predict(test_df).values

y_true = model_df["y"].values
y_pred = oof_probs.argmax(axis=1)  #

print(f"Out-of-Fold Accuracy: {accuracy_score(y_true, y_pred):.3f}")
print(f"Out-of-Fold Log-Loss: {log_loss(y_true, oof_probs):.4f}")

for train_idx, test_idx in gkf_state.split(model_df, model_df["y"], groups=model_df["inputstate"]):
    fold_res = smf.mnlogit(formula, data=model_df.iloc[train_idx]).fit(method="bfgs", maxiter=500, disp=False)
    state_oof_probs[test_idx] = fold_res.predict(model_df.iloc[test_idx]).values

print(f"Leave-States-Out Accuracy: {accuracy_score(y_true, state_oof_probs.argmax(axis=1)):.3f}")
print(f"Leave-States-Out Log-Loss: {log_loss(y_true, state_oof_probs):.4f}")

Out-of-Fold Accuracy: 0.903
Out-of-Fold Log-Loss: 0.2890
Leave-States-Out Accuracy: 0.896
Leave-States-Out Log-Loss: 0.3081


Confusion matrix & classification report

In [11]:
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Cand 1", "Cand 2", "Cand 3"], zero_division=0))


Confusion Matrix (rows=actual, cols=predicted):
[[32850  2479     0]
 [ 3158 23037     0]
 [  270    98     0]]

Classification Report:
              precision    recall  f1-score   support

      Cand 1       0.91      0.93      0.92     35329
      Cand 2       0.90      0.88      0.89     26195
      Cand 3       0.00      0.00      0.00       368

    accuracy                           0.90     61892
   macro avg       0.60      0.60      0.60     61892
weighted avg       0.90      0.90      0.90     61892



Seat-by-seat vote share error

In [12]:
eval_df = model_df[["inputstate", "seat", "y", "commonweight"]].reset_index(drop=True).copy()
for k in range(len(valid_slots)):
    eval_df[f"pred_{k}"] = oof_probs[:, k]
    eval_df[f"actual_{k}"] = (eval_df["y"] == k).astype(float)

seat_eval = (
    eval_df.groupby(["inputstate", "seat"])
    .apply(lambda g: pd.Series({
        "n_resp": len(g),
        "pred_c1": np.average(g["pred_0"], weights=g["commonweight"]),
        "actual_c1": np.average(g["actual_0"], weights=g["commonweight"]),
        "pred_c2": np.average(g["pred_1"], weights=g["commonweight"]),
        "actual_c2": np.average(g["actual_1"], weights=g["commonweight"]),
    }))
    .reset_index()
)

seat_eval["c1_abs_err"] = (seat_eval["pred_c1"] - seat_eval["actual_c1"]).abs()
seat_eval["c2_abs_err"] = (seat_eval["pred_c2"] - seat_eval["actual_c2"]).abs()

print("\nMean Absolute Vote-Share Error (percentage points) by Seat:")
print(
    seat_eval.groupby("seat")[["c1_abs_err", "c2_abs_err"]]
    .mean()
    .mul(100)
    .round(2)
)


Mean Absolute Vote-Share Error (percentage points) by Seat:
          c1_abs_err  c2_abs_err
seat                            
CC22_365        7.77        6.39
CC22_366        5.92        5.36
